In [88]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import deque
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt


device = 'cuda'

In [89]:
# Replay Buffer for storing experiences
import torch
import torch.nn as nn
import numpy as np
import random
from collections import deque


device = 'cuda'


class ReplayBuffer():
    def __init__(self, buffer_size):
        self.memory = deque(maxlen=buffer_size)

    def add(self, state, action, next_state, reward):
        self.memory.append((state, action, next_state, reward))

    def sample(self, batch_size):
        batch = random.sample(self.memory, batch_size)
        states, actions, next_states, rewards = zip(*batch)
        return states, actions, next_states, rewards

    def __len__(self):
        return len(self.memory)

# Policy Network (Actor)
class policy_net(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim, num_layers=8):
        super(policy_net, self).__init__()
        layers = [nn.Linear(state_dim, hidden_dim), nn.ReLU()]
        for _ in range(num_layers - 2):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.ReLU()])
        layers.extend([nn.Linear(hidden_dim, action_dim), nn.Tanh()])
        self.model = nn.Sequential(*layers)
    
    def forward(self, state):
        return self.model(state)

# Q-Network (Critic)
class Q_net(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim, num_layers=8):
        super(Q_net, self).__init__()
        self.ln1 = nn.Linear(state_dim, hidden_dim)
        layers = [nn.Linear(hidden_dim + action_dim, hidden_dim), nn.ReLU()]
        for _ in range(num_layers - 2):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.ReLU()])
        layers.append(nn.Linear(hidden_dim, 1))
        self.model = nn.Sequential(*layers)
    
    def forward(self, state, action):
        x = torch.relu(self.ln1(state))
        x = torch.cat([x, action], dim=1)
        return self.model(x)

# DDPG Agent
class ddpg():
    def __init__(self, state_dim, action_dim, hidden_dim, policy_lr=1e-3, Q_lr=1e-3, batch_size=32, buffer_size=1000, tau=5e-3, gamma=0.99):
        # Initialize networks
        self.policy = policy_net(state_dim,action_dim, hidden_dim).to(device)
        self.policy_target = policy_net(state_dim, action_dim, hidden_dim).to(device)
        self.Q_net = Q_net(state_dim, action_dim, hidden_dim).to(device)
        self.Q_net_target = Q_net(state_dim, action_dim, hidden_dim).to(device)
        
        # Optimizers
        self.policy_optimizer = torch.optim.Adam(self.policy.parameters(), lr=policy_lr)
        self.Q_net_optimizer = torch.optim.Adam(self.Q_net.parameters(), lr=Q_lr)
        
        # Replay Buffer
        self.memory = ReplayBuffer(buffer_size)

        #other hyperparameters
        self.batch_size = batch_size
        self.tau = tau
        self.gamma = gamma
        
        # Action noise for exploration
        self.noise = torch.tensor([[0.0]], dtype=torch.float32, device=device)
        
        # Initialize target networks to match current networks
        self._update_target(self.policy, self.policy_target, tau=1)
        self._update_target(self.Q_net, self.Q_net_target, tau=1)
    
    def act(self, state):
        action = self.policy(state).detach()
        return torch.clamp(action ,min=-1, max=1)
    
    def add(self, state, action, next_state, reward):
        self.memory.add(state, action, next_state, reward)
    
    def learn(self, batch_size=32):
        batch_size = batch_size
        
        # Sample batch from replay buffer
        state, action, next_state, reward = self.memory.sample(batch_size)
        
        # Convert to torch tensors
        state = torch.cat(state)
        action = torch.cat(action)
        reward = torch.cat(reward)
        
        # Compute Q-value estimates
        state_action_values = self.Q_net(state, action).squeeze(1)
        
        # Mask for non-terminal states
        non_final_mask = torch.tensor(tuple(map(lambda s: s is not None, next_state)), dtype=torch.bool, device=device)
        non_final_next_states = torch.cat([s for s in next_state if s is not None])
        
        
        # Compute next state values
        next_state_values = torch.zeros(batch_size, device=device)
        next_action = self.policy_target(non_final_next_states)
        
        with torch.no_grad():
            next_state_values[non_final_mask] = self.Q_net_target(non_final_next_states, next_action).squeeze(1)
        
        # Compute target Q-values
        target_state_action_values = next_state_values * self.gamma + reward
        
        # Define loss function (MSE Loss for Q-network)
        criterion = nn.MSELoss()
        loss = criterion(state_action_values, target_state_action_values)
        
        # Update Q-network
        self.Q_net_optimizer.zero_grad()
        loss.backward()
        self.Q_net_optimizer.step()
        
        # Update policy network
        self.policy_optimizer.zero_grad()
        J = -self.Q_net(state, self.policy(state)).mean()  # Policy loss
        J.backward()
        self.policy_optimizer.step()
        
        # Soft update target networks
        self._update_target(self.Q_net, self.Q_net_target, self.tau)
        self._update_target(self.policy, self.policy_target, self.tau)
    
    def _update_target(self, current, target, tau):
        current_dict = current.state_dict()
        target_dict = target.state_dict()
        for key in current_dict:
            target_dict[key] = current_dict[key] * tau + target_dict[key] * (1 - tau)
        target.load_state_dict(target_dict)

# Training Function

def train(agent, env, episodes=100, updates=1, steps=100):
    for i_episode in range(episodes):
        # Reset environment
        state, _ = env.reset()
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        rewards =[]
        for step in range(steps):
            # Agent selects action
            action = agent.act(state=state)
            
            # Execute action in environment
            # print(state)
            # print(action)
            observation, reward, terminated, truncated, _ = env.step(action)
            reward = torch.tensor([reward], device=device)
            
            # Handle terminal state
            next_state = None if terminated else observation
            
            # Store experience in replay buffer
            agent.add(state, action, next_state, reward)
            
            # Update state
            state = next_state
            
            # Train if memory has enough samples
            if len(agent.memory) >= agent.batch_size:
                for update in range(updates):
                    agent.learn()
            rewards.append(reward)
            if terminated or truncated:
                break
        print(f'episode {i_episode} finished')
        print(sum(rewards[-10:])/10)
        print('------------------------------')
        



In [90]:
class objectiveFunction():
    def __init__(self):
        self.A = torch.rand((10,10), dtype=torch.float32, device=device)
        self.b = torch.rand(10, dtype=torch.float32, device=device)

    def __call__(self, x):
        out = torch.norm(self.A @ x - self.b, p=2 )
        return out.item()

class Environment(gym.Env):
    """Custom Gymnasium environment with continuous state and action spaces."""
    
    def __init__(self, objective):
        super(Environment, self).__init__()

        # Continuous state space: Vector of 3 values in range [-5, 5]
        self.state_dim = 10
        self.observation_space = spaces.Box(low=float('-inf'), high=float('inf'), shape=(self.state_dim,), dtype=np.float32)

        # Continuous action space: Vector of 2 values in range [-1, 1]
        self.action_dim = 10
        self.action_space = spaces.Box(low=float('-inf'), high=float('inf'), shape=(self.action_dim,), dtype=np.float32)

        # Initialize state
        self.state = torch.rand(10, dtype=torch.float32, device=device)
        self.max_steps = 10  # Limit number of steps
        self.current_step = 0

        #objective fucntion
        self.reward_func = objective

    def reset(self, seed=None, options=None):
        """ Reset the environment and return initial state. """
        super().reset(seed=seed)
        self.state = torch.rand(10, dtype=torch.float32, device=device)
        self.current_step = 0
        return self.state, {}

    def step(self, action):
        """ Apply action, update state, compute reward, and return observations. """
    
    
        # Ensure action is within bounds
    
        # Example state transition: Move state based on action
        self.state += action.squeeze(0)   # Modify state with action influence
        

        # Define reward function: Example: Negative distance to target (0,0,0)
        reward = -self.reward_func(self.state)

        # Episode termination conditions
        self.current_step += 1
        done = abs(reward) < 0.1  # Done if close to origin
        truncated = self.current_step >= self.max_steps  # Done if max steps reached

        return self.state.unsqueeze(0), reward, done, truncated, {}

    def render(self):
        """ Render current state (simple print for now). """
        print(f"Step: {self.current_step}, State: {self.state}")

    def close(self):
        """ Clean up resources (if needed). """
        pass


In [91]:
objective = objectiveFunction()
env = Environment(objective)


In [92]:
A = objective.A
b = objective.b

x = torch.linalg.lstsq(A,b).solution
torch.norm(A @ x -b)


tensor(4.4129e-07, device='cuda:0')

In [93]:
state, _ = env.reset()
state_dim = 10
action_dim =  10
hidden_dim = 512
policy_net_lr = 1e-3
Q_net_lr = 1e-3
buffer_size = 500
batch_size = 128
tau = 5e-3
gamma = 0.99
episodes = 1000
updates = 1
steps = 100

agent = ddpg(state_dim, action_dim, hidden_dim, policy_net_lr, Q_net_lr, batch_size, buffer_size, tau, gamma)
train(agent,env,episodes,updates,steps)

/tmp/ipykernel_294494/804219020.py:152: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)


episode 0 finished
tensor([-4.9098], device='cuda:0')
------------------------------
episode 1 finished
tensor([-3.2519], device='cuda:0')
------------------------------
episode 2 finished
tensor([-4.4280], device='cuda:0')
------------------------------
episode 3 finished
tensor([-5.9575], device='cuda:0')
------------------------------
episode 4 finished
tensor([-6.4894], device='cuda:0')
------------------------------
episode 5 finished
tensor([-4.8401], device='cuda:0')
------------------------------
episode 6 finished
tensor([-2.1618], device='cuda:0')
------------------------------
episode 7 finished
tensor([-7.5975], device='cuda:0')
------------------------------
episode 8 finished
tensor([-4.4272], device='cuda:0')
------------------------------
episode 9 finished
tensor([-8.3318], device='cuda:0')
------------------------------
episode 10 finished
tensor([-3.9301], device='cuda:0')
------------------------------
episode 11 finished
tensor([-3.7822], device='cuda:0')
---------

KeyboardInterrupt: 